# Steady turn trim examples

このNotebookでは、VSPAEROの`.stab`線形空力モデルを使い、定常旋回トリムを次の順に計算する。

1. 動力・高度維持・協調旋回
2. 動力・高度維持・ラダーのみ旋回
3. 無動力・定常降下・協調旋回
4. 無動力・ラダー限界旋回

すべての角度は内部でrad、角速度はrad/sとして扱う。表示時にdegまたはdeg/sへ変換する。

このNotebookでは`G103A.stab`の参照寸法と速度をSI単位として扱い、質量と密度もSI単位で明示する。`.stab`に保存された`Rho`は自動変換されないため、ここでは`rho=None`を使わない。


## 1. importとリポジトリ位置

Notebookの実行場所から親ディレクトリを順に調べ、`src/TrimTurnSolver.py`を含むリポジトリルートを見つける。


In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import sys

import pandas as pd
from IPython.display import display

start_dir = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in [start_dir, *start_dir.parents]
        if (candidate / "src" / "TrimTurnSolver.py").exists()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the repository root containing "
        "src/TrimTurnSolver.py."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.TrimTurnSolver import (  # noqa: E402
    read_vspaero_stab,
    resolve_control_columns_from_stab,
    solve_rudder_limit_turn,
    solve_steady_gliding_turn,
    solve_steady_level_turn,
)

print("repo_root:", repo_root)


## 2. 入力モデルと共通条件

通常変更する設定をこのセルへまとめる。

`G103A.stab`の`Rho`はこのexampleで使うSI密度と一致しないため、solverへ`rho=1.225`を明示する。


In [ ]:
stab_path = (
    repo_root
    / "examples"
    / "models"
    / "G103A"
    / "G103A.stab"
)

mass = 580.0
rho = 1.225
speed = 30.0
turn_rate = 0.06

if not stab_path.exists():
    raise FileNotFoundError(stab_path)

stab = read_vspaero_stab(stab_path)
control_columns = resolve_control_columns_from_stab(stab)

model_summary = pd.Series(
    {
        "stab_path": str(stab_path),
        "Sref": stab.Sref,
        "Cref": stab.Cref,
        "Bref": stab.Bref,
        "Vinf": stab.V0,
        "Rho_in_stab": stab.rho0,
        "rho_used_by_example": rho,
        "control_columns": control_columns,
    },
    name="value",
)
display(model_summary)


## 3. 結果表示

同じ形式のトリム結果を複数回表示するため、解の単位変換だけを共通化する。


In [ ]:
ANGLE_PARAMETERS = {
    "alpha",
    "beta",
    "phi",
    "theta",
    "delta_e",
    "delta_a",
    "delta_r",
}


def turn_solution_table(result: dict) -> pd.DataFrame:
    rows = []

    for name, value in result["solution"].items():
        row = {
            "parameter": name,
            "value": float(value),
            "unit": "",
            "converted_value": float("nan"),
            "converted_unit": "",
        }

        if name in ANGLE_PARAMETERS:
            row["unit"] = "rad"
            row["converted_value"] = math.degrees(value)
            row["converted_unit"] = "deg"
        elif name == "Omega":
            row["unit"] = "rad/s"
            row["converted_value"] = math.degrees(value)
            row["converted_unit"] = "deg/s"
        elif name == "V":
            row["unit"] = "m/s"
            row["converted_value"] = 3.6 * value
            row["converted_unit"] = "km/h"
        elif name == "T":
            row["unit"] = "N"

        rows.append(row)

    return pd.DataFrame(rows)


## 4. 動力・高度維持・協調旋回

固定条件は、

$$
V=30\ \mathrm{m/s},
\qquad
\Omega=0.06\ \mathrm{rad/s},
\qquad
\beta=0
$$

とする。

速度、旋回率、横滑りなしを指定し、必要な姿勢角、舵角、推力を解く。


In [ ]:
level_coordinated_turn = solve_steady_level_turn(
    fixed={
        "V": speed,
        "Omega": turn_rate,
        "beta": 0.0,
    },
    stab_path=stab_path,
    mass=mass,
    rho=rho,
    initial_guess={
        "alpha": math.radians(2.0),
        "phi": math.radians(10.0),
        "theta": math.radians(2.0),
    },
    verbose=1,
)

display(
    pd.Series(
        {
            "passed": level_coordinated_turn["passed"],
            "max_abs_residual": level_coordinated_turn[
                "max_abs_residual"
            ],
            "turn_radius": level_coordinated_turn[
                "derived"
            ]["turn_radius"],
            "sink_rate": level_coordinated_turn[
                "derived"
            ]["sink_rate"],
        },
        name="value",
    )
)
display(turn_solution_table(level_coordinated_turn))
display(
    pd.Series(
        level_coordinated_turn["residuals"],
        name="residual",
    )
)


## 5. 動力・高度維持・ラダーのみ旋回

固定条件は、

$$
V=30\ \mathrm{m/s},
\qquad
\Omega=0.06\ \mathrm{rad/s},
\qquad
\delta_a=0
$$

とする。

エルロンを使わずに指定旋回率を維持するため、横滑り角とラダー舵角は結果として求める。


In [ ]:
level_rudder_only_turn = solve_steady_level_turn(
    fixed={
        "V": speed,
        "Omega": turn_rate,
        "delta_a": 0.0,
    },
    stab_path=stab_path,
    mass=mass,
    rho=rho,
    initial_guess={
        "alpha": math.radians(2.0),
        "beta": math.radians(2.0),
        "phi": math.radians(10.0),
        "theta": math.radians(2.0),
    },
    verbose=1,
)

display(
    pd.Series(
        {
            "passed": level_rudder_only_turn["passed"],
            "max_abs_residual": level_rudder_only_turn[
                "max_abs_residual"
            ],
            "turn_radius": level_rudder_only_turn[
                "derived"
            ]["turn_radius"],
            "sink_rate": level_rudder_only_turn[
                "derived"
            ]["sink_rate"],
        },
        name="value",
    )
)
display(turn_solution_table(level_rudder_only_turn))
display(
    pd.Series(
        level_rudder_only_turn["residuals"],
        name="residual",
    )
)


## 6. 無動力・定常降下・協調旋回

固定条件は、

$$
V=30\ \mathrm{m/s},
\qquad
\Omega=0.06\ \mathrm{rad/s},
\qquad
\beta=0
$$

とする。

滑空旋回では$T=0$をsolver内部で固定し、高度維持条件は課さない。得られた状態から`h_dot`と`sink_rate`を後処理する。


In [ ]:
gliding_coordinated_turn = solve_steady_gliding_turn(
    fixed={
        "V": speed,
        "Omega": turn_rate,
        "beta": 0.0,
    },
    stab_path=stab_path,
    mass=mass,
    rho=rho,
    initial_guess={
        "alpha": math.radians(2.0),
        "phi": math.radians(10.0),
        "theta": math.radians(0.0),
    },
    verbose=1,
)

display(
    pd.Series(
        {
            "passed": gliding_coordinated_turn["passed"],
            "max_abs_residual": gliding_coordinated_turn[
                "max_abs_residual"
            ],
            "turn_radius": gliding_coordinated_turn[
                "derived"
            ]["turn_radius"],
            "h_dot": gliding_coordinated_turn[
                "derived"
            ]["h_dot"],
            "sink_rate": gliding_coordinated_turn[
                "derived"
            ]["sink_rate"],
        },
        name="value",
    )
)
display(turn_solution_table(gliding_coordinated_turn))
display(
    pd.Series(
        gliding_coordinated_turn["residuals"],
        name="residual",
    )
)


## 7. 無動力・ラダー限界旋回

`solve_rudder_limit_turn()`は、

$$
\delta_r=\pm\delta_{r,\max}
$$

の両端を解き、受理された解のうち最大の$|\phi|$を選ぶ。

この`.stab`は局所線形モデルであるため、exampleでは比較的小さいラダー限界$\delta_{r,\max}=0.5^\circ$を使う。より大きな限界を調べる場合は、`passed`と残差を必ず確認する。


In [ ]:
rudder_limit_turn = solve_rudder_limit_turn(
    stab_path=stab_path,
    mass=mass,
    delta_r_max=math.radians(0.5),
    mode="gliding",
    V=speed,
    delta_a=0.0,
    rho=rho,
    initial_guess={
        "alpha": gliding_coordinated_turn[
            "solution"
        ]["alpha"],
        "beta": 0.0,
        "phi": gliding_coordinated_turn[
            "solution"
        ]["phi"],
        "theta": gliding_coordinated_turn[
            "solution"
        ]["theta"],
        "Omega": gliding_coordinated_turn[
            "solution"
        ]["Omega"],
        "delta_e": gliding_coordinated_turn[
            "solution"
        ]["delta_e"],
    },
    verbose=1,
)

display(
    pd.Series(
        {
            "passed": rudder_limit_turn["passed"],
            "message": rudder_limit_turn["message"],
            "selected_side": rudder_limit_turn[
                "selected_side"
            ],
            "delta_r_max_deg": math.degrees(
                rudder_limit_turn["delta_r_max"]
            ),
            "limiting_delta_r_deg": (
                math.degrees(
                    rudder_limit_turn[
                        "limiting_delta_r"
                    ]
                )
                if rudder_limit_turn["passed"]
                else float("nan")
            ),
            "max_abs_phi_deg": (
                math.degrees(
                    rudder_limit_turn["max_abs_phi"]
                )
                if rudder_limit_turn["passed"]
                else float("nan")
            ),
        },
        name="value",
    )
)

if rudder_limit_turn["passed"]:
    display(turn_solution_table(rudder_limit_turn))
    display(
        pd.Series(
            rudder_limit_turn["residuals"],
            name="residual",
        )
    )


## 8. 確認事項

各caseについて、少なくとも次を確認する。

- `passed=True`
- `max_abs_residual`が`residual_tol`以下
- 協調旋回では`beta=0`
- ラダーのみ旋回では`delta_a=0`
- 滑空旋回では`T=0`
- `turn_radius`、`sink_rate`、舵角が想定する範囲内

`.stab`の一次線形空力モデルから大きく離れる迎角、横滑り角、舵角が得られた場合は、solverが収束していても空力モデルの適用範囲を別途確認する。
